In [ ]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import time
import re
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("StealthPhisher2025_Labelled.csv")
LABEL = df.iloc[:,-1:].columns[0]

print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
df.columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="white")  # Remove grid
plt.rcParams.update({
    'font.weight': 'bold',        # Make all text bold
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,        # Slightly larger titles
    'axes.labelsize': 12,        # Larger axis labels
    'legend.title_fontsize': 12, # Bold legend title
    'legend.fontsize': 10        # Adjust legend font size
})

# Function to create side-by-side histograms
def plot_side_by_side_histogram(df, feature, label_column='Label', save_path=None):
    """
    Generates side-by-side histograms for a feature split by label categories.
    :param df: DataFrame to plot from
    :param feature: Feature to plot
    :param label_column: Column representing the label (e.g., 'Label')
    :param save_path: Path to save the graph (optional)
    """
    # Create figure
    plt.figure(figsize=(8, 5))

    # Custom color palette: blue for Legitimate, orange for Phishing
    custom_palette = {"Legitimate": "lime", "Phishing": "gold"}

    # Plot histograms with hue (side-by-side)
    sns.histplot(
        data=df,
        x=feature,
        hue=label_column,
        multiple="dodge",  # Side-by-side bars
        palette=custom_palette,
        edgecolor="black",
        alpha=0.7
    )

    # Add titles and labels
    plt.xlabel(feature.replace("_", " ").title(), fontsize=14, fontweight='bold')
    plt.ylabel("Count", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')

    # Add a border around the plot
    ax = plt.gca()  # Get current axis
    for spine in ax.spines.values():
        spine.set_edgecolor('black')  # Set border color
        spine.set_linewidth(1)  # Set border width

    # **Remove this line if you want borders on all sides**
    # sns.despine(top=False, right=False)  # Keep all spines

    # Save the figure if needed
    if save_path:
        plt.savefig(save_path, format='png', dpi=500, bbox_inches='tight')

    # Show the plot
    plt.tight_layout()
    plt.show()

selected_features = ['LengthOfURL', 'WAPLegitimate', 'WAPPhishing', 'ShannonEntropy', 'KolmogorovComplexity']

for feature in selected_features:
    save_path = f"{feature}_SideBySide_Distribution.png"
    plot_side_by_side_histogram(df, feature, save_path=save_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

selected_features = [
    'LengthOfURL', 'UniqueFeatureCnt', 'FractalDimension', 'KolmogorovComplexity', 
    'HexPatternCnt', 'Base64PatternCnt', 'LikelinessIndex'
]

# Function to create KDE plots for smoother density curves
def plot_stacked_kde(df, feature, label_column='Label', save_path=None):
    """
    Generates stacked KDE plots for a feature split by label categories.
    :param df: DataFrame to plot from
    :param feature: Feature to plot
    :param label_column: Column representing the label (e.g., 'Label')
    :param save_path: Path to save the graph (optional)
    """
    # Define a custom color palette for the hue categories
    custom_palette = {'Legitimate': 'lime', 'Phishing': 'gold'}

    # Create figure
    plt.figure(figsize=(8, 5))

    # KDE plot
    sns.kdeplot(
        data=df,
        x=feature,
        hue=label_column,
        fill=True,  # Stacked KDE
        alpha=0.6,
        palette=custom_palette  # Apply custom color palette
    )

    # Add titles and labels
    plt.xlabel(feature.replace("_", " ").title(), fontsize=14, fontweight='bold')
    plt.ylabel("Density", fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')

    # Add a border around the plot
    ax = plt.gca()  # Get current axis
    for spine in ax.spines.values():
        spine.set_edgecolor('black')  # Set border color
        spine.set_linewidth(1)  # Set border width

    # Remove this line if you want to retain all borders
    # sns.despine(top=False, right=False)  # Ensure all spines are visible

    # Save the figure if needed
    if save_path:
        plt.savefig(save_path, format='png', dpi=500, bbox_inches='tight')

    # Show the plot
    plt.tight_layout()
    plt.show()

# Example: Plot for all selected features
for feature in selected_features:
    save_path = f"{feature}_StackedKDE_Distribution.png"
    plot_stacked_kde(df, feature, save_path=save_path)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# General settings for seaborn and matplotlib
sns.set(style="white")  # Remove grid
plt.rcParams.update({
    'font.weight': 'bold',        # Make all text bold
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,        # Slightly larger titles
    'axes.labelsize': 12,        # Larger axis labels
    'legend.title_fontsize': 12, # Bold legend title
    'legend.fontsize': 10        # Adjust legend font size
})

# Function to add a border around the plot
def add_plot_border(ax):
    """Ensure all sides of the plot have visible borders."""
    for spine in ax.spines.values():
        spine.set_edgecolor('black')  # Set border color
        spine.set_linewidth(1)        # Set border width

# Scatterplot of Fractal Dimension vs. Kolmogorov Complexity
plt.figure(figsize=(8, 5))
scatter_ax = sns.scatterplot(
    data=df, 
    x='FractalDimension', 
    y='KolmogorovComplexity', 
    hue='Label', 
    palette={'Legitimate': 'lime', 'Phishing': 'gold'},  # Explicit color mapping
    alpha=0.6
)
plt.xlabel('Fractal Dimension', fontsize=12, weight='bold')
plt.ylabel('Kolmogorov Complexity', fontsize=12, weight='bold')
plt.legend(title='Label', loc='best', title_fontsize=12, fontsize=10)  # Bold legend title
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')

# Add border to the scatterplot
add_plot_border(plt.gca())

plt.savefig("kolmogorovcomplexity.png", format='png', dpi=500, bbox_inches='tight')
plt.show()

# Density Plot of Likeliness Index by Label
plt.figure(figsize=(8, 5))
density_ax = sns.kdeplot(
    data=df, 
    x='LikelinessIndex', 
    hue='Label', 
    fill=True, 
    common_norm=False, 
    palette={'Legitimate': 'lime', 'Phishing': 'gold'},  # Explicit color mapping
    alpha=0.6
)
plt.xlabel('Likeliness Index', fontsize=12, weight='bold')
plt.ylabel('Density', fontsize=12, weight='bold')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')

# Add border to the density plot
add_plot_border(plt.gca())

plt.savefig("likelinessindex.png", format='png', dpi=500, bbox_inches='tight')
plt.show()

# Pairplot of selected features
selected_features = ['WAPLegitimate', 'WAPPhishing', 'ShannonEntropy', 'FractalDimension', 'KolmogorovComplexity', 'Label']
pairplot = sns.pairplot(
    df[selected_features], 
    hue='Label', 
    palette={'Legitimate': 'lime', 'Phishing': 'gold'},  # Explicit color mapping
    diag_kind="kde"
)

# Add a border to each axis in the pairplot
for ax in pairplot.axes.flat:
    if ax:  # Some axes might be empty (e.g., for the diagonal)
        add_plot_border(ax)

plt.subplots_adjust(top=0.95)  # Adjust spacing for suptitle
pairplot.savefig("ds_pairplot_features.png", format='png', dpi=500, bbox_inches='tight')
plt.show()